# Asset retrieval validation

Does text and local graph context retrieve records of the same structured asset after identifier and template controls?

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.data import write_json

pd.set_option("display.max_columns", 20)

from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize
from src.data import validate_equipment_id, sha256
from src.retrieval import mask_identifiers, normalize_template, build_graph_features, fit_representations, rank_candidates
from src.evaluation import evaluate_rankings, random_expectation, bootstrap_ci, paired_comparisons, lift_intervals

findings = json.loads((ROOT / "results/retrieval/diagnostic/findings.json").read_text(encoding="utf-8"))
print(f"Previous diagnostic: {findings['random_template_overlap']} shared random-split templates; "
      f"failure labels derived from query text: {findings['failure_labels_derived_from_query_text']}.")
records = pd.read_csv(ROOT / "results/text/record_audit.csv", keep_default_na=False)
entities = pd.read_csv(ROOT / "results/text/entities.csv", keep_default_na=False)
graph_edges = pd.read_csv(ROOT / "results/kg/edges.csv", keep_default_na=False)
out = ROOT / "results/retrieval/asset"
out.mkdir(parents=True, exist_ok=True)

Previous diagnostic: 55 shared random-split templates; failure labels derived from query text: True.


<local environment>:27: DtypeWarning: Columns (3: row_id, 9: span_start, 10: span_end) have mixed types. Specify dtype option on import or set low_memory=False.
  graph_edges = pd.read_csv(ROOT / "results/kg/edges.csv", keep_default_na=False)


## Protocol

Notebook 05's saved audit motivates replacing the text-derived failure target with structured Equipment_ID and excluding identifier and exact-template shortcuts. We compare every prespecified method, including the failure-mode ablation, without selecting thresholds or a preferred method from the results.

This is retrieval of records for known synthetic assets, not prediction for unseen assets. Query records are held out of the index; assets may occur in both sets. Equipment_ID is an independent field, not independently verified physical ground truth.

In [2]:
protocol = {
    "seed": config.SEED, "target": "structured Equipment_ID", "minimum_asset_records": config.MIN_ASSET_RECORDS,
    "minimum_templates_per_asset": 2, "query_cap": config.MAX_QUERIES,
    "top_k": list(config.TOP_K), "bootstrap_repetitions": config.BOOTSTRAPS,
    "bootstrap_unit": "equipment_id", "fit_representations": "candidate records only",
    "rrf_k": 60, "exclude_same_template": True, "exclude_same_work_order": True,
    "queries_without_positives": "retain in unconditional metrics; also report answerable-only metrics",
    "selection": "no GO threshold and no tuned winner",
    "diagnostic_findings_sha256": sha256(ROOT / "results/retrieval/diagnostic/findings.json")
}
write_json(out / "protocol.json", protocol)
validated = records["equipment_id_raw"].map(validate_equipment_id)
assert validated.equals(records["equipment_id"])
id_audit = pd.DataFrame([{"input_records": len(records), "valid_id_records": int(validated.ne("").sum()),
                          "invalid_id_records": int(validated.eq("").sum()), "valid_assets": validated[validated.ne("")].nunique()}])
fields = ["equipment_id", "order_type", "maintenance_type", "problem_text", "action_text", "combined_text"]
conflicts = records.groupby("work_order")[fields].nunique().gt(1).any(axis=1)
conflicting_orders = conflicts.index[conflicts]
conflict_records = records.loc[records["work_order"].isin(conflicting_orders)]
base = records.loc[~records["work_order"].isin(conflicting_orders) & records["equipment_id"].ne("")].copy()
base = base.sort_values("row_id").drop_duplicates("work_order")
base = base.loc[base["combined_text"].ne("")].copy()
base["masked_text"] = base["combined_text"].map(mask_identifiers)
base["template"] = base["combined_text"].map(normalize_template)
counts = base["equipment_id"].value_counts()
templates = base.groupby("equipment_id")["template"].nunique()
assets = counts[counts.ge(config.MIN_ASSET_RECORDS)].index.intersection(templates[templates.ge(2)].index)
cohort = base.loc[base["equipment_id"].isin(assets)].sort_values("row_id").reset_index(drop=True)
assert len(cohort) > config.MAX_QUERIES
id_audit.to_csv(out / "id_audit.csv", index=False)
conflict_records.to_csv(out / "conflicting_work_orders.csv", index=False)
display(id_audit)
display(pd.DataFrame([{"cohort_records": len(cohort), "cohort_assets": cohort["equipment_id"].nunique(),
                       "conflicting_work_orders_excluded": len(conflicting_orders), "templates": cohort["template"].nunique()}]))

,input_records,valid_id_records,invalid_id_records,valid_assets
0,48850,47562,1288,9400


,cohort_records,cohort_assets,conflicting_work_orders_excluded,templates
0,38534,5128,10,3051


## Query and candidate separation

In [3]:
queries = cohort.sample(n=config.MAX_QUERIES, random_state=config.SEED).sort_values("row_id").reset_index(drop=True)
candidates = cohort.loc[~cohort["row_id"].isin(queries["row_id"])].reset_index(drop=True)
assert not set(candidates["row_id"]) & set(queries["row_id"])
assert not set(candidates["work_order"]) & set(queries["work_order"])
assert not queries["masked_text"].str.contains(r"\d").any()
assert not candidates["masked_text"].str.contains(r"\d").any()
random = random_expectation(queries, candidates)
queries = queries.merge(random[["query_id", "n_candidates", "positives"]], left_on="row_id", right_on="query_id", validate="one_to_one")
cohort[["row_id", "work_order", "equipment_id", "masked_text", "template"]].to_csv(out / "cohort.csv", index=False)
candidates[["row_id", "work_order", "equipment_id", "masked_text", "template"]].to_csv(out / "candidates.csv", index=False)
queries[["row_id", "work_order", "equipment_id", "masked_text", "template", "n_candidates", "positives"]].to_csv(out / "queries.csv", index=False)
display(queries[["n_candidates", "positives"]].describe())

,n_candidates,positives
count,1500.000000,1500.000000
mean,36748.360000,8.128667
std,208.804433,5.958505
min,36370.000000,1.000000
25%,36542.000000,4.000000
50%,36799.500000,6.000000
75%,36914.000000,11.000000
max,37034.000000,31.000000


## Representations and complete eligible rankings

Graph context uses one-hop lexical entities plus order and maintenance types. Equipment nodes, asset mentions, work-order IDs and graph paths through asset targets are excluded. Vocabulary and IDF are fitted only on candidates; ties use a seeded permutation independent of labels.

In [4]:
candidate_full = build_graph_features(candidates, graph_edges)
query_full = build_graph_features(queries, graph_edges)
candidate_no_failure = build_graph_features(candidates, graph_edges, include_failure=False)
query_no_failure = build_graph_features(queries, graph_edges, include_failure=False)
all_tokens = {token for features in candidate_full + query_full for token in features}
assert all(token.split(":", 1)[0] in {"equipment_type", "component", "maintenance_action", "failure_mode", "order_type", "maintenance_type"} for token in all_tokens)
assert not any("failure_mode:" in token for features in candidate_no_failure + query_no_failure for token in features)
ct, qt, cg, qg = fit_representations(candidates["masked_text"], queries["masked_text"], candidate_full, query_full)
no_failure_vectorizer = DictVectorizer(sparse=True, sort=True)
cn = normalize(no_failure_vectorizer.fit_transform(candidate_no_failure))
qn = normalize(no_failure_vectorizer.transform(query_no_failure))
rankings, hybrid, pool_audit = rank_candidates([ct, cn, cg], [qt, qn, qg], candidates, queries, exclude_templates=True)
assert pool_audit[["self_matches", "same_work_order_matches", "same_template_matches"]].eq(0).all().all()
assert (pool_audit["n_candidates"].to_numpy() == random["n_candidates"].to_numpy()).all()
pool_audit.to_csv(out / "candidate_audit.csv", index=False)
methods = {"Masked TF-IDF": rankings[0], "Graph context without failure_mode": rankings[1],
           "Graph context full": rankings[2], "Hybrid RRF": hybrid}
results = pd.concat([evaluate_rankings(ranking, queries, candidates, "equipment_id", method)
                     for method, ranking in methods.items()], ignore_index=True)
results = pd.concat([results, random.drop(columns=["n_candidates", "positives"])], ignore_index=True)
metric_cols = [f"hit@{k}" for k in config.TOP_K] + ["mrr@10", "mrr@50"]
metrics = results.groupby("method")[metric_cols].mean().reset_index()
display(metrics.round(6))

,method,hit@1,hit@5,hit@10,hit@20,hit@50,mrr@10,mrr@50
0,Graph context full,0.002000,0.010000,0.022667,0.044000,0.075333,0.006049,0.008430
1,Graph context without failure_mode,0.001333,0.008667,0.020667,0.039333,0.076667,0.004923,0.007334
2,Hybrid RRF,0.001333,0.010667,0.033333,0.048000,0.111333,0.006875,0.009785
3,Masked TF-IDF,0.004000,0.016667,0.033333,0.066000,0.135333,0.010198,0.014559
4,Random expectation,0.000222,0.001108,0.002214,0.004421,0.011001,0.000649,0.000994


## Uncertainty and ablation

Asset-cluster bootstrap resamples assets rather than treating related records as independent. Shared resamples support paired method differences and lift intervals. These intervals condition on this fixed sample, index and query selection; they do not establish generalization to real plants.

In [5]:
ci = bootstrap_ci(results, metric_cols)
pairs = [("Graph context full", "Graph context without failure_mode"),
         ("Graph context full", "Masked TF-IDF"), ("Hybrid RRF", "Masked TF-IDF"),
         ("Hybrid RRF", "Graph context full")]
differences = paired_comparisons(results, metric_cols, pairs)
lift = lift_intervals(results, metric_cols)
answerable_ids = random.loc[random["positives"].gt(0), "query_id"]
answerable_metrics = results.loc[results["query_id"].isin(answerable_ids)].groupby("method")[metric_cols].mean().reset_index()
results.to_csv(out / "per_query.csv", index=False)
random.to_csv(out / "random_expectation.csv", index=False)
metrics.to_csv(out / "metrics.csv", index=False)
ci.to_csv(out / "confidence_intervals.csv", index=False)
differences.to_csv(out / "paired_differences.csv", index=False)
lift.to_csv(out / "lift.csv", index=False)
answerable_metrics.to_csv(out / "answerable_metrics.csv", index=False)
display(ci.loc[ci["metric"].isin(["hit@10", "mrr@50"])].round(6))
display(differences.loc[differences["metric"].isin(["hit@10", "mrr@50"])].round(6))
display(lift.loc[lift["metric"].eq("hit@10")].round(3))

,method,metric,estimate,ci95_low,ci95_high,clusters,resampling_unit
2,Graph context full,hit@10,0.022667,0.015452,0.030224,1246,equipment_id
6,Graph context full,mrr@50,0.008430,0.005800,0.011556,1246,equipment_id
9,Graph context without failure_mode,hit@10,0.020667,0.014309,0.028015,1246,equipment_id
13,Graph context without failure_mode,mrr@50,0.007334,0.005158,0.010208,1246,equipment_id
16,Hybrid RRF,hit@10,0.033333,0.025164,0.042477,1246,equipment_id
20,Hybrid RRF,mrr@50,0.009785,0.007429,0.012695,1246,equipment_id
23,Masked TF-IDF,hit@10,0.033333,0.024724,0.042343,1246,equipment_id
27,Masked TF-IDF,mrr@50,0.014559,0.011171,0.018940,1246,equipment_id
30,Random expectation,hit@10,0.002214,0.002109,0.002328,1246,equipment_id
34,Random expectation,mrr@50,0.000994,0.000948,0.001046,1246,equipment_id


,left,right,metric,difference,ci95_low,ci95_high
2,Graph context full,Graph context without failure_mode,hit@10,0.002000,-0.000673,0.005338
6,Graph context full,Graph context without failure_mode,mrr@50,0.001096,0.000019,0.002633
9,Graph context full,Masked TF-IDF,hit@10,-0.010667,-0.019718,-0.001312
13,Graph context full,Masked TF-IDF,mrr@50,-0.006129,-0.010603,-0.002446
16,Hybrid RRF,Masked TF-IDF,hit@10,0.000000,-0.006337,0.007258
20,Hybrid RRF,Masked TF-IDF,mrr@50,-0.004774,-0.008094,-0.002083
23,Hybrid RRF,Graph context full,hit@10,0.010667,0.003987,0.018018
27,Hybrid RRF,Graph context full,mrr@50,0.001355,-0.000689,0.003131


,method,metric,lift,ci95_low,ci95_high
2,Graph context full,hit@10,10.238,7.027,13.612
9,Graph context without failure_mode,hit@10,9.335,6.386,12.685
16,Hybrid RRF,hit@10,15.056,11.314,19.455
23,Masked TF-IDF,hit@10,15.056,11.055,19.215


## Final result

Interpret absolute hit rates alongside the number of assets, candidate-pool size and lift over exact random retrieval. A large lift can coexist with low usefulness. Failure-mode ablation and paired differences determine what these data support; no method is assumed superior. This experiment does not validate the cross-source ontology mapping.

In [6]:
checks = {
    "structured_target_validated": validated.equals(records["equipment_id"]),
    "query_candidate_rows_disjoint": not bool(set(candidates["row_id"]) & set(queries["row_id"])),
    "query_candidate_work_orders_disjoint": not bool(set(candidates["work_order"]) & set(queries["work_order"])),
    "numeric_identifiers_masked": not queries["masked_text"].str.contains(r"\d").any(),
    "target_and_asset_mentions_excluded_from_graph_features": all(token.split(":", 1)[0] in {"equipment_type", "component", "maintenance_action", "failure_mode", "order_type", "maintenance_type"} for token in all_tokens),
    "same_template_candidates_excluded": pool_audit["same_template_matches"].eq(0).all(),
    "complete_eligible_candidate_pool": (pool_audit["n_candidates"].to_numpy() == random["n_candidates"].to_numpy()).all(),
    "graph_ablation_without_failure_mode": not any("failure_mode:" in token for features in candidate_no_failure for token in features)
}
checks_frame = pd.DataFrame([{"check": key, "passed": bool(value)} for key, value in checks.items()])
assert checks_frame["passed"].all()
checks_frame.to_csv(out / "leakage_checks.csv", index=False)
summary = {
    "cohort_records": len(cohort), "cohort_assets": int(cohort["equipment_id"].nunique()),
    "candidate_records": len(candidates), "candidate_assets": int(candidates["equipment_id"].nunique()),
    "query_records": len(queries), "query_assets": int(queries["equipment_id"].nunique()),
    "answerable_queries": int(random["positives"].gt(0).sum()),
    "median_candidate_pool": float(random["n_candidates"].median()),
    "median_positives": float(random["positives"].median()),
    "valid_assets_before_cohort": int(validated[validated.ne("")].nunique()),
    "conflicting_work_orders_excluded": len(conflicting_orders),
    "representation": "candidate-fitted TF-IDF and one-hop entity context; known-asset record retrieval",
    "metrics": metrics.to_dict("records"), "checks_passed": int(checks_frame["passed"].sum()),
    "checks_total": len(checks_frame), "bootstrap_unit": "asset", "bootstrap_repetitions": config.BOOTSTRAPS,
    "scope": "synthetic known-asset retrieval only; no real-plant or cross-source validation"
}
write_json(out / "summary.json", summary)
display(checks_frame)
display(pd.DataFrame([{key: value for key, value in summary.items() if key != "metrics"}]))

,check,passed
0,structured_target_validated,True
1,query_candidate_rows_disjoint,True
2,query_candidate_work_orders_disjoint,True
3,numeric_identifiers_masked,True
4,target_and_asset_mentions_excluded_from_graph_...,True
5,same_template_candidates_excluded,True
6,complete_eligible_candidate_pool,True
7,graph_ablation_without_failure_mode,True


,cohort_records,cohort_assets,candidate_records,candidate_assets,query_records,query_assets,answerable_queries,median_candidate_pool,median_positives,valid_assets_before_cohort,conflicting_work_orders_excluded,representation,checks_passed,checks_total,bootstrap_unit,bootstrap_repetitions,scope
0,38534,5128,37034,5128,1500,1246,1500,36799.5,6.0,9400,10,candidate-fitted TF-IDF and one-hop entity con...,8,8,asset,500,synthetic known-asset retrieval only; no real-...
